# PHẦN VII — POST-DEPLOY VERIFY (RUNBOOK RÚT GỌN)

Notebook rút gọn: chỉ lệnh + comment ngắn.

## BƯỚC 28 — Load admin-openrc

In [ ]:
source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

env | grep OS_
which openstack
openstack --version

## BƯỚC 28 — Verify Keystone/services

In [ ]:
openstack token issue
openstack service list
openstack endpoint list
openstack service list | egrep "identity|compute|network|image|volume|placement" || true

## BƯỚC 28 — Verify Nova

In [ ]:
openstack compute service list
openstack hypervisor list
openstack hypervisor stats show

## BƯỚC 28 — Verify Neutron/OVN

In [ ]:
openstack network agent list || true
docker ps | egrep "neutron|ovn|openvswitch" || true
ansible -i /etc/kolla/multinode all -m shell -a "docker ps --format '{{.Names}}' | egrep 'neutron|ovn|openvswitch' || true" 

## BƯỚC 28 — Verify Cinder/Glance

In [ ]:
openstack volume service list
openstack image list

docker ps | grep cinder || true
docker ps | grep glance || true

## BƯỚC 28 — Test Ceph volume backend

In [ ]:
openstack volume create --size 1 test-vol
openstack volume list
openstack volume show test-vol

sudo rbd -n client.cinder -p volumes ls || true
sudo rbd -p volumes ls || true

openstack volume delete test-vol
openstack volume list

## BƯỚC 29 — Download Ubuntu 22.04 image

In [ ]:
mkdir -p ~/openstack-images
cd ~/openstack-images

wget -c https://cloud-images.ubuntu.com/jammy/current/jammy-server-cloudimg-amd64.img
ls -lh jammy-server-cloudimg-amd64.img
qemu-img info jammy-server-cloudimg-amd64.img || true

## BƯỚC 29 — Upload image

In [ ]:
source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh
cd ~/openstack-images

openstack image create \
  --file jammy-server-cloudimg-amd64.img \
  --disk-format qcow2 \
  --container-format bare \
  --public \
  --property hw_disk_bus=virtio \
  --property hw_vif_model=virtio \
  "Ubuntu 22.04 LTS"

openstack image list
sudo rbd -n client.glance -p images ls || true

## BƯỚC 29 — Tạo flavors

In [ ]:
openstack flavor create --vcpus 1 --ram 1024 --disk 10 tiny
openstack flavor create --vcpus 2 --ram 2048 --disk 20 small
openstack flavor create --vcpus 4 --ram 4096 --disk 40 medium

openstack flavor list

## BƯỚC 29 — Tạo public network/subnet

In [ ]:
openstack network create \
  --external \
  --provider-network-type flat \
  --provider-physical-network physnet1 \
  public

openstack subnet create \
  --network public \
  --subnet-range 192.168.150.200/29 \
  --allocation-pool start=192.168.150.201,end=192.168.150.206 \
  --gateway 192.168.150.254 \
  --no-dhcp \
  public-subnet

openstack network list
openstack subnet list

## BƯỚC 30 — Tạo script 12 tenants

In [ ]:
cat > ~/init-tenants.sh << 'EOF'
#!/bin/bash
set -e

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

for i in $(seq -w 01 12); do
  PROJECT="project-hv${i}"
  USER="hv${i}"
  PASS="HocVien${i}@Lab2025!"

  openstack project show "$PROJECT" >/dev/null 2>&1 || \
    openstack project create --description "Lab project cho HV${i}" --enable "$PROJECT"

  openstack user show "$USER" >/dev/null 2>&1 || \
    openstack user create --password "$PASS" --email "hv${i}@lab.local" --enable "$USER"

  openstack role add --project "$PROJECT" --user "$USER" member

  openstack quota set \
    --cores 8 --ram 16384 --instances 5 \
    --volumes 5 --gigabytes 100 \
    --floating-ips 3 --networks 3 --routers 2 \
    "$PROJECT"

  echo "✓ $USER / $PROJECT / $PASS"
done

openstack project list | grep hv || true
openstack user list | grep hv || true
EOF

chmod +x ~/init-tenants.sh
ls -lh ~/init-tenants.sh

## BƯỚC 30 — Chạy script tạo tenant

In [ ]:
~/init-tenants.sh

## BƯỚC 30 — Verify tenants/quota

In [ ]:
openstack project list | grep hv
openstack user list | grep hv
openstack role assignment list --user hv01 --project project-hv01 --names
openstack quota show project-hv01

for i in $(seq -w 01 12); do
  echo "===== project-hv${i} ====="
  openstack quota show project-hv${i} | egrep "cores|ram|instances|volumes|gigabytes|floating|networks|routers"
done

## BƯỚC 30 — Test login hv01

In [ ]:
cat > ~/hv01-openrc.sh << EOF
export OS_AUTH_URL=${OS_AUTH_URL}
export OS_PROJECT_NAME=project-hv01
export OS_USERNAME=hv01
export OS_PASSWORD='HocVien01@Lab2025!'
export OS_USER_DOMAIN_NAME=Default
export OS_PROJECT_DOMAIN_NAME=Default
export OS_IDENTITY_API_VERSION=3
EOF

source ~/hv01-openrc.sh
openstack token issue
openstack server list
openstack volume list
openstack network list

source /etc/kolla/admin-openrc.sh